# Day 3 · 더러운 데이터 세탁 (데이터 전처리)
AICE 2주 합격 플랜 · 영상 3편과 함께 풉니다. 60분, 손으로 직접 치세요.
- 순서는 공식 샘플과 같습니다: **결측치 → 이상치 → 인코딩 → 스케일링**
- 이 노트북은 **해설**입니다.

### 준비: 연습 데이터 내려받기
아래 셀을 먼저 실행하세요. 같은 폴더에 `aice_practice.csv` 가 생깁니다.

In [ ]:
!wget -q -O aice_practice.csv https://raw.githubusercontent.com/pm-ainote/aice/main/data/aice_practice.csv
print('준비 완료')

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
df = pd.read_csv('aice_practice.csv')
df.shape

## 1. [직접 작성] 컬럼별 결측치 개수를 출력하고, 전체 결측치 개수를 `답안1` 에 저장하세요

In [ ]:
print(df.isnull().sum())
답안1 = df.isnull().sum().sum()
답안1

## 2. [생각해 보기] `dropna()` 를 하면 몇 행이 남는지 `답안2` 에 저장하세요 (df 는 바꾸지 마세요)
몇 퍼센트를 버리게 되는지도 계산해 보세요.

In [ ]:
답안2 = len(df.dropna())
print(답안2, f'{1 - 답안2 / len(df):.0%} 손실')

## 3. [직접 작성] Age 결측치를 Age 의 중앙값으로 채우세요
- 채운 뒤 Age 결측치 개수를 `답안3` 에 저장하세요.

In [ ]:
df['Age'] = df['Age'].fillna(df['Age'].median())
답안3 = df['Age'].isnull().sum()
답안3

## 4. [오류 정정] Embarked 를 최빈값으로 채웠는데 결측치가 그대로 남습니다 (에러는 안 남)
- 고친 뒤 Embarked 결측치 개수를 `답안4` 에 저장하세요.

In [ ]:
# 아래의 코드를 정정하고 실행하세요.
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])   # mode() 는 시리즈
답안4 = df['Embarked'].isnull().sum()
답안4

## 5. [빈칸 채우기] Fare 의 IQR 울타리를 구하세요

In [ ]:
# 아래 코드의 빈칸을 채우고 실행하세요.
q1 = df['Fare'].quantile(0.25)
q3 = df['Fare'].quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
low, high

## 6. [직접 작성] 울타리 밖 행의 Pclass 분포를 출력하고, 그중 1등석 개수를 `답안6` 에 저장하세요
지우기 전에 무엇을 지우는지 먼저 봅니다.

In [ ]:
out = df[(df['Fare'] < low) | (df['Fare'] > high)]
print(out['Pclass'].value_counts())
답안6 = (out['Pclass'] == 1).sum()
답안6

## 7. [직접 작성] 울타리 안쪽 행만 남겨 `df_iqr` 에 저장하고, 행 개수를 `답안7` 에 저장하세요

In [ ]:
df_iqr = df[(df['Fare'] >= low) & (df['Fare'] <= high)]
답안7 = len(df_iqr)
답안7

## 8. [직접 작성] df 에서 글자(object) 컬럼 이름을 리스트로 `답안8` 에 저장하세요

In [ ]:
답안8 = df.select_dtypes('object').columns.tolist()
답안8

## 9. [직접 작성] Sex 를 male 0, female 1 로 바꾸세요
- 바꾼 뒤 Sex 의 합(여성 수)을 `답안9` 에 저장하세요.

In [ ]:
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
답안9 = df['Sex'].sum()
답안9

## 10. [빈칸 채우기] Embarked 를 원 핫 인코딩하고, 첫 범주 열은 빼세요
- 결과를 `df_enc` 에 저장하고 컬럼 이름을 확인하세요.

In [ ]:
# 아래 코드의 빈칸을 채우고 실행하세요.
df_enc = pd.get_dummies(df, columns=['Embarked'], drop_first=True)
df_enc.columns.tolist()

## 11. [오류 정정] 스케일링 코드에 평가를 망치는 실수가 있습니다 (공식 샘플 9번 형식)
- 에러는 안 납니다. 무엇이 잘못인지 찾아 고치세요.

In [ ]:
X = df_enc[['Age', 'Fare']]
y = df_enc['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 아래의 코드를 정정하고 실행하세요.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)   # 시험용은 transform 만
X_train.mean(axis=0).round(2), X_test.mean(axis=0).round(2)

## 12. [직접 작성] RobustScaler 로 같은 작업을 하고, 학습용 Fare 의 중앙값(center_)을 소수 둘째 자리까지 `답안12` 에 저장하세요

In [ ]:
X_train2, X_test2 = train_test_split(X, test_size=0.2, random_state=42)
rs = RobustScaler()
X_train2 = rs.fit_transform(X_train2)
X_test2 = rs.transform(X_test2)
답안12 = round(rs.center_[1], 2)
답안12

---
### 오늘의 체크
- [ ] 숫자는 중앙값, 글자는 mode()[0]
- [ ] 처리한 뒤 isnull().sum() 으로 다시 확인
- [ ] IQR 울타리: q1 - 1.5*IQR, q3 + 1.5*IQR, 지우기 전에 무엇인지 본다
- [ ] 값 2개는 map, 3개 이상은 get_dummies(drop_first=True)
- [ ] 학습용은 fit_transform, 시험용은 transform